# 🐟 Fish Audio S2 Pro - Colab 一键极速部署

本笔记本经过深度优化，专为 Colab T4 GPU 环境设计。

### 🚀 快速开始
1. **启用 GPU**: `Runtime` -> `Change runtime type` -> `T4 GPU`。
2. **设置 Token (可选)**: 点击左侧 🔑 图标 (Secrets)，添加一个名为 `HF_TOKEN` 的变量，填入你的 Hugging Face Token 并开启 "Notebook access"。这能显著加快模型下载速度。
3. **一键运行**: `Runtime` -> `Run all`。

In [ ]:
# @title 1. 环境配置与代码拉取
import os
from google.colab import userdata

REPO_URL = "https://github.com/entishl/fish-s2-pro-colab.git"
REPO_DIR = "fish-s2-pro-colab"

# 尝试获取 HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ 成功从 Secrets 读取 HF_TOKEN")
except Exception:
    print("ℹ️ 未在 Secrets 中发现 HF_TOKEN，将使用匿名下载 (速度较慢)")

if not os.path.exists(REPO_DIR):
    print(f"正在初始化仓库: {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("正在更新代码...")
    %cd {REPO_DIR}
    !git pull
    %cd ..

%cd {REPO_DIR}

print("正在安装系统依赖...")
!apt-get update && apt-get install -y ffmpeg libsox-dev portaudio19-dev

print("正在安装 uv 包管理器...")
!pip install uv

print("正在使用 uv 配置隔离环境并极速安装依赖 (约需 1 分钟)...")
# 1. 创建名为 .venv 的虚拟环境
!uv venv
# 2. 使用官方推荐的 cu124 版本进行源码安装，极大减少冲突可能
!uv pip install -e ".[cu124]"
# 3. 安装所有的原项目依赖
!uv pip install -r requirements.txt
# 4. 补充 Gradio
!uv pip install gradio

In [ ]:
# @title 2. 启动应用
import sys
import os

# 必须在虚拟环境中启动 app.py 才能避开 Colab 预装冲突
print("正在独立的虚拟环境中启动应用...")
!./.venv/bin/python app.py

### 💡 小贴士
- **模型加载**: 第一次运行会从 Hugging Face 下载约 9GB 的权重，请耐心等待。
- **公共链接**: 运行成功后，点击输出末尾的 `public URL: https://xxxxxxxx.gradio.live` 即可开始使用。